# Deployment dan Monitoring — Random Forest Classifier

Notebook ini disesuaikan untuk proyek **Social Media Impact on Life** yang memakai `RandomForestClassifier`.

## Sebelum mulai
1. Buka notebook modelling lama `laporan_lab.ipynb`.
2. Pastikan cell training Random Forest berhasil dijalankan dan variabel `model` adalah pipeline yang sudah dilatih.
3. Jalankan **Cell 1** di notebook modelling lama untuk menyimpan model.
4. Pastikan file `random_forest_bundle.joblib` berada di folder yang sama dengan `deployment.ipynb` dan `app.py`.
5. Jalankan cell berikutnya satu per satu.

> Jangan menyimpan model Gradient Boosting Regressor (`gbr_model`) dari materi lama. Kita harus memakai model Random Forest Classifier hasil training kemarin.

## Cell 1 — Simpan model terlatih (jalankan di `laporan_lab.ipynb`)

**Penting:** cell ini dijalankan di notebook modelling lama, bukan di notebook deployment ini, karena variabel `model` berada di notebook modelling.

Model disimpan bersama daftar fitur yang benar-benar dipakai saat training. Dengan begitu, API bisa menerima fitur yang sama saat prediksi.

In [2]:
import joblib

# Pastikan variabel `model` adalah pipeline Random Forest yang sudah dilatih.
# Pipeline mencakup preprocessing dan classifier.
feature_columns = model.feature_names_in_.tolist()

bundle = {
    "model": model,
    "feature_columns": feature_columns
}

joblib.dump(bundle, "random_forest_bundle.joblib")
print("Model dan daftar fitur berhasil disimpan.")
print("Jumlah fitur:", len(feature_columns))
print("Fitur:", feature_columns)

NameError: name 'model' is not defined

Setelah cell di atas berhasil, salin `random_forest_bundle.joblib` ke folder yang sama dengan file `deployment.ipynb` ini (kalau belum berada di folder yang sama).

## Cell 2 — Buat aplikasi Flask (`app.py`)

Jalankan cell ini. Kode di bawah akan membuat file `app.py` di folder kerja notebook.

In [ ]:
%%writefile app.py
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)

# Memuat pipeline Random Forest dan fitur yang digunakan saat training
bundle = joblib.load("random_forest_bundle.joblib")
model = bundle["model"]
feature_columns = bundle["feature_columns"]

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": "Random Forest Classifier"})

@app.route("/predict", methods=["POST"])
def predict():
    payload = request.get_json(silent=True)

    if not isinstance(payload, dict) or "data" not in payload:
        return jsonify({
            "error": "Kirim JSON dengan format {'data': {...}} atau {'data': [{...}]}"
        }), 400

    data = payload["data"]

    # Satu baris boleh dikirim sebagai object; beberapa baris sebagai list of objects
    if isinstance(data, dict):
        data = [data]

    if not isinstance(data, list) or len(data) == 0 or not all(isinstance(row, dict) for row in data):
        return jsonify({"error": "'data' harus berupa object atau list berisi object"}), 400

    df = pd.DataFrame(data)

    # Periksa fitur yang wajib ada agar input cocok dengan model training
    missing = [col for col in feature_columns if col not in df.columns]
    if missing:
        return jsonify({
            "error": "Ada fitur wajib yang belum dikirim",
            "missing_features": missing,
            "required_features": feature_columns
        }), 400

    # Susun kolom sesuai urutan saat training; kolom tambahan diabaikan
    df = df[feature_columns]

    try:
        predictions = model.predict(df)
        return jsonify({"prediction": [str(value) for value in predictions]})
    except Exception as exc:
        return jsonify({"error": "Prediksi gagal", "detail": str(exc)}), 400

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000, debug=False)

## Cell 3 — Jalankan server

Setelah Cell 2 berhasil membuat `app.py`:

1. Buka **Terminal → New Terminal** di VS Code.
2. Pastikan terminal berada di folder yang berisi `app.py` dan `random_forest_bundle.joblib`.
3. Aktifkan environment Python yang biasa dipakai untuk praktikum, jika belum aktif.
4. Instal library yang dibutuhkan jika belum tersedia:

```bash
python -m pip install flask joblib pandas scikit-learn requests
```

5. Jalankan server:

```bash
python app.py
```

Biarkan terminal tetap berjalan. Server seharusnya aktif di `http://127.0.0.1:5000`. Jangan menjalankan perintah `python app.py` di cell Python biasa; jalankan di terminal.

## Cell 4 — Uji endpoint dari notebook

Jalankan cell ini **setelah server Flask aktif di terminal**. Contoh ini mengambil satu baris dari dataset, lalu mengirim fitur yang diperlukan ke API. Pastikan file CSV ada di folder yang sama; ubah nama/path file jika berbeda.

In [ ]:
import joblib
import pandas as pd
import requests

# Memuat daftar fitur dari bundle yang sama dengan aplikasi Flask
bundle = joblib.load("random_forest_bundle.joblib")
feature_columns = bundle["feature_columns"]

# Baca dataset yang dipakai pada praktikum
df_test_api = pd.read_csv("Social_media_impact_on_life.csv")

# Ambil satu contoh baris dan gunakan hanya fitur yang diperlukan model
sample = df_test_api[feature_columns].iloc[0].to_dict()

response = requests.post(
    "http://127.0.0.1:5000/predict",
    json={"data": sample},
    timeout=10
)

print("HTTP status:", response.status_code)
print("Response:", response.json())

## Monitoring sederhana

Endpoint `/health` dapat digunakan untuk mengecek apakah server merespons. Jalankan cell berikut ketika server masih aktif.

In [ ]:
health_response = requests.get("http://127.0.0.1:5000/health", timeout=10)
print("HTTP status:", health_response.status_code)
print("Response:", health_response.json())

## Catatan penting
- Prediksi API harus menggunakan fitur yang sama dengan model saat training.
- File `random_forest_bundle.joblib` adalah artefak model; jangan menggantinya dengan file model regresi dari materi lama.
- Endpoint `/health` hanya memeriksa apakah aplikasi merespons, belum memantau perubahan kualitas prediksi dari waktu ke waktu.
- Untuk monitoring kualitas model secara nyata, simpan input, prediksi, dan label aktual (jika tersedia), lalu evaluasi ulang accuracy/F1 secara berkala.